In [ ]:
import os
import xarray as xr
import pandas as pd

def calculate_average_pr(file_path, start_date, end_date):
    dataset = xr.open_dataset(file_path)
    print(f"Processing {file_path}")  # Debug line

    if 'pr' in dataset.data_vars:
        pr_data = dataset['pr'].sel(time=slice(start_date, end_date))

        if pr_data.size > 0:
            return pr_data.mean().values
        else:
            print(f"No data found for {file_path} from {start_date} to {end_date}")  # Debug line
            return None
    else:
        print(f"'pr' variable not found in {file_path}")  # Debug line
        return None

def calculate_plot_index(data):
    def plot_index(row):
        model = row['Model']
        ecoregion = row['Ecoregion']
        term = row['Term']
        current_value = row['Ave25yearSpan_Annual_Pr']
        
        past_value = data[(data['Model'] == model) & 
                          (data['Ecoregion'] == ecoregion) & 
                          (data['Term'] == 'near-term Past')]['Ave25yearSpan_Annual_Pr']
        
        if not past_value.empty:
            past_value = past_value.iloc[0]
            if past_value != 0:  # Prevent division by zero
                return ((current_value - past_value) / past_value) * 100
        return None

    data['Plot_index_Annual_Pr'] = data.apply(plot_index, axis=1)

def main(main_directory):
    results = []

    time_periods = {
        "historical": ("1990-01-01", "2014-12-31"),
        "ssp126": {
            "near-term Future": ("2025-01-01", "2049-12-31"),
            "mid-term Future": ("2050-01-01", "2074-12-31"),
            "long-term Future": ("2075-01-01", "2099-12-31"),
        },
        "ssp245": {
            "near-term Future": ("2025-01-01", "2049-12-31"),
            "mid-term Future": ("2050-01-01", "2074-12-31"),
            "long-term Future": ("2075-01-01", "2099-12-31"),
        },
        "ssp370": {
            "near-term Future": ("2025-01-01", "2049-12-31"),
            "mid-term Future": ("2050-01-01", "2074-12-31"),
            "long-term Future": ("2075-01-01", "2099-12-31"),
        },
        "ssp585": {
            "near-term Future": ("2025-01-01", "2049-12-31"),
            "mid-term Future": ("2050-01-01", "2074-12-31"),
            "long-term Future": ("2075-01-01", "2099-12-31"),
        },
    }

    for root, dirs, files in os.walk(main_directory):
        for file in files:
            if file.endswith('.nc'):
                model = file.split('_')[6]
                file_name_without_ext = os.path.splitext(file)[0]
                eco_region = file_name_without_ext.split('_')[-1]
            
                if 'historical' in file.lower():
                    start_date, end_date = time_periods["historical"]
                    avg_pr = calculate_average_pr(os.path.join(root, file), start_date, end_date)
                    if avg_pr is not None:
                        results.append(["near-term Past", model, eco_region, "historical", avg_pr])

                for scenario in ["ssp126", "ssp245", "ssp370", "ssp585"]:
                    if scenario in file.lower():
                        for term, (start_date, end_date) in time_periods[scenario].items():
                            avg_pr = calculate_average_pr(os.path.join(root, file), start_date, end_date)
                            if avg_pr is not None:
                                results.append([term, model, eco_region, scenario, avg_pr])

    df = pd.DataFrame(results, columns=["Term", "Model", "Ecoregion", "Scenario", "Ave25yearSpan_Annual_Pr"])
    
    # Calculate the Plot_index_Annual_Pr
    calculate_plot_index(df)

    df.to_csv("1-Ave25yearSpan_Annual_Pr.csv", index=False)
    print("Output saved to 1-Ave25yearSpan_Annual_Pr.csv")

if __name__ == "__main__":
    main_directory = r"E:\MajidDehghaniSanij\ExtremeProjections\Datasets\4-EcoClip"
    main(main_directory)
